In [23]:
import arcpy
import os

In [24]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Input bathymetry rasters (* make sure to use no overlap bathy raster for GBC)
gbc_bathy = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\nooverlap_GBC15a_bathy_p26920_v6642"
usc_bathy = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\USC10m_bathy_p26920_v6642"

# Input slope rasters (* make sure to use no overlap slope raster for GBC)
gbc_slope = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\nooverlap_GBC15a_slope_p26920_v6642"
usc_slope = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\USC10m_slope_p26920_v6642"

# --------

# Geodatabase for output mosaic datasets
gdb = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb"

# Output standalone rasters created from mosaics
final_bathy_raster = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\BATHY_USC10mGBC15a_p26920_v6642"
final_slope_raster = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\SLOPE_USC10mGBC15a_p26920_v6642"

In [25]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun
arcpy.env.compression = "NONE"      # set compression to NONE for output mosaics and rasters

# set environment coordinate system: horizontal EPSG:26920 + vertical EPSG:6642 (coordinate system is set within Create Mosaic Dataset tool and Copy Raster *should* carry over vertical coordinate system from the input mosaic, but set coordinate system for entire environment just to be safe and ensure output merged rasters include vertical coordinate systems)
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(26920, 6642)

In [26]:
# ── 1. CREATE MOSAIC DATASETS FOR BATHYMETRY AND SLOPE ───────────────────────────────────────────────

# Create Mosaic Dataset (Data Management Tools): Bathymetry
arcpy.management.CreateMosaicDataset(
    in_workspace=gdb,
    in_mosaicdataset_name="mosaic_bathy_USC10mGBC15a_p26920_v6642",
    coordinate_system=arcpy.SpatialReference(26920, 6642),
    num_bands=1,
    # USC bathy is 32-bit float but GBC bathy is 16-bit short - use wider pixel type of the two rasters to avoid truncating USC decimal depth values
    pixel_type="32_BIT_FLOAT"
)
# define path to new mosaic for use in future blocks + print confirmation mosaic was successfully created
mosaic_bathy = os.path.join(gdb, "mosaic_bathy_USC10mGBC15a_p26920_v6642")
print(f"Created mosaic dataset: {mosaic_bathy} \n")

# Create Mosaic Dataset (Data Management Tools): Slope
arcpy.management.CreateMosaicDataset(
    in_workspace=gdb,
    in_mosaicdataset_name="mosaic_slope_USC10mGBC15a_p26920_v6642",
    coordinate_system=arcpy.SpatialReference(26920, 6642),
    num_bands=1,
    # both input rasters are 32-bit float
    pixel_type="32_BIT_FLOAT"
)
# define path to new mosaic for use in future blocks + print confirmation mosaic was successfully created
mosaic_slope = os.path.join(gdb, "mosaic_slope_USC10mGBC15a_p26920_v6642")
print(f"Created mosaic dataset: {mosaic_slope} \n")

Created mosaic dataset: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\mosaic_bathy_USC10mGBC15a_p26920_v6642 

Created mosaic dataset: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\mosaic_slope_USC10mGBC15a_p26920_v6642 



In [27]:
# ── 2. ADD USC AND GBC RASTERS TO MOSAICS ────────────────────────────────────────────────────────────

# Add Rasters to Mosaic Dataset (Data Management Tools): Bathymetry
arcpy.management.AddRastersToMosaicDataset(
    in_mosaic_dataset=mosaic_bathy,
    raster_type="Raster Dataset",
    input_path=[gbc_bathy, usc_bathy],
    # sets range each item (USC, GCB) based on native resolution of input item (so sets USC range centered around 10m and GBC range centered around ~450m)
    update_cellsize_ranges="UPDATE_CELL_SIZES",
    update_boundary="UPDATE_BOUNDARY",
    update_overviews="NO_OVERVIEWS",
    build_pyramids="BUILD_PYRAMIDS",
    calculate_statistics="CALCULATE_STATISTICS",
    # matches on raster path so only overwrites if re-adding exact same file paths
    duplicate_items_action="OVERWRITE_DUPLICATES"
)
# print confirmation both rasters were added to mosaic
print(f"Added {os.path.basename(gbc_bathy)} and {os.path.basename(usc_bathy)} rasters to {mosaic_bathy}")

# Add Rasters to Mosaic Dataset (Data Management Tools): Slope
arcpy.management.AddRastersToMosaicDataset(
    in_mosaic_dataset=mosaic_slope,
    raster_type="Raster Dataset",
    input_path=[gbc_slope, usc_slope],
    # sets range each item (USC, GCB) based on native resolution of input item (so sets USC range centered around 10m and GBC range centered around ~450m)
    update_cellsize_ranges="UPDATE_CELL_SIZES",
    update_boundary="UPDATE_BOUNDARY",
    update_overviews="NO_OVERVIEWS",
    build_pyramids="BUILD_PYRAMIDS",
    calculate_statistics="CALCULATE_STATISTICS",
    # matches on raster path so only overwrites if re-adding exact same file paths
    duplicate_items_action="OVERWRITE_DUPLICATES"
)
# print confirmation both rasters were added to mosaic
print(f"\n Added {os.path.basename(gbc_slope)} and {os.path.basename(usc_slope)} rasters to {mosaic_slope} \n")

Added nooverlap_GBC15a_bathy_p26920_v6642 and USC10m_bathy_p26920_v6642 rasters to C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\mosaic_bathy_USC10mGBC15a_p26920_v6642

 Added nooverlap_GBC15a_slope_p26920_v6642 and USC10m_slope_p26920_v6642 rasters to C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\mosaic_slope_USC10mGBC15a_p26920_v6642 



In [28]:
# ── SETTINGS FOR RASTER EXPORT ───────────────────────────────────────────────────────────────────────

# Set mosaics to export with cell size = 10 m so USC portion of final output rasters keeps native 10 m resolution (GBC portion of output rasters is resampled from native ~450 m to 10 m resolution in final output rasters only, NOT in mosaics or input rasters)
arcpy.env.cellSize = "10"

# Use bilinear resampling (ideal method for continuous data like depth/slope) for any cells where the output grid doesn't line up one-to-one with the cell in the source grid (due to different cell size/offset grid origin/reprojection/etc)
# ** ONLY GBC CELLS GET RESAMPLED (CELL SIZE INPUT 450 M ≠ OUTPUT 10 M) 
# ** USC CELLS ARE NOT RESAMPLED BECAUSE THEY ALREADY HAVE SAME CELL SIZE AS OUTPUT (10 M) + SNAP RASTER LINE BELOW ENSURES OUTPUT CELL BOUNDARIES ALIGN EXACTLY WITH INPUT (SO USC VALUES PASS THROUGH UNCHANGED TO FINAL OUTPUT RASTER)
arcpy.env.resamplingMethod = "BILINEAR"

# 'Copy Raster' tool default will drop items whose pixel size range (set by "update_cellsize_ranges" line in 'Add Rasters to Mosaic Dataset' tool) does NOT include the requested output cell size (10m) - increase cell size tolerance by a factor of 1000 to avoid dropping GBC item whose ~450m-centered range does not include 10m requested output (increases valid native size to 1000x smaller/larger to encompass requested output size)
arcpy.management.SetMosaicDatasetProperties(in_mosaic_dataset=mosaic_bathy, cell_size_tolerance=1000)
arcpy.management.SetMosaicDatasetProperties(in_mosaic_dataset=mosaic_slope, cell_size_tolerance=1000)

<Result 'C:\\Users\\kkeii\\OneDrive\\Documents\\ArcGIS\\Files\\THESIS_FILES\\Geodatabases\\Bathymetry_geodatabases\\MASTER_RASTERS_USC_GBC_MERGED.gdb\\mosaic_slope_USC10mGBC15a_p26920_v6642'>

In [29]:
# ── 3. EXPORT EACH MOSAIC TO SINGLE STANDALONE RASTER ────────────────────────────────────────────────

######### bathymetry #########

# * snap output bathy raster to input USC bathy raster grid so USC cells pass through unchanged (will NOT change position of GBC raster cells)
arcpy.env.snapRaster = usc_bathy

# Copy Raster (Data Management Tools): Bathymetry
arcpy.management.CopyRaster(
    in_raster=mosaic_bathy,
    out_rasterdataset=final_bathy_raster,
    # use wider pixel type of the two rasters (USC 32-bit float vs GBC 16-bit short) to avoid truncating USC decimal depth values
    pixel_type="32_BIT_FLOAT"
)
print(f"Exported bathymetry raster: {final_bathy_raster} \n")

# calculate statistics of new output bathy raster (ensures accurate min/max values are printed by print block below)
arcpy.management.CalculateStatistics(in_raster_dataset=final_bathy_raster)

######### slope #########

# * snap output slope raster to input USC slope raster grid so USC cells pass through unchanged (will NOT change position of GBC raster cells)
arcpy.env.snapRaster = usc_slope

# Copy Raster (Data Management Tools): Slope
arcpy.management.CopyRaster(
    in_raster=mosaic_slope,
    out_rasterdataset=final_slope_raster,
    # both input rasters are 32-bit float
    pixel_type="32_BIT_FLOAT"
)
print(f"Exported slope raster: {final_slope_raster} \n")

# calculate statistics of new output slope raster (ensures accurate min/max values are printed by print block below)
arcpy.management.CalculateStatistics(in_raster_dataset=final_slope_raster)

######### check outputs #########

# Print names and ranges of each output raster to check data is valid
for path in [
    # input bathymetry rasters: USC min = -6239.1694, max = 0**, cell size = 10** / GBC min = -8619**, max = 0**, cell size = 472.85
    final_bathy_raster,
    # input slope rasters: USC min = 0**, max = 89.6427**, cell size = 10** / GBC min = 0**, max = 74.0227, cell size = 472.85
    final_slope_raster,
]:
    r = arcpy.Raster(path)
    print(f"{os.path.basename(path)}: min = {r.minimum}, max = {r.maximum}, cellsize = {r.meanCellWidth} \n")

Exported bathymetry raster: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\BATHY_USC10mGBC15a_p26920_v6642 

Exported slope raster: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\SLOPE_USC10mGBC15a_p26920_v6642 

BATHY_USC10mGBC15a_p26920_v6642: min = -8617.681640625, max = 0.0, cellsize = 10.0 

SLOPE_USC10mGBC15a_p26920_v6642: min = 0.0, max = 89.42605590820312, cellsize = 10.0 



In [30]:
# clear the environment overrides (so they do not leak into later cells/runs)
arcpy.ClearEnvironment("cellSize")
arcpy.ClearEnvironment("resamplingMethod")
arcpy.ClearEnvironment("snapRaster")

# reset coordinate system
arcpy.ClearEnvironment("outputCoordinateSystem")